# Airport Bird & Drone Detector — Training Notebook

This notebook trains a **MobileNetV2 binary image classifier** to distinguish birds from drones,
then exports the model for real-time browser inference via **TensorFlow.js**.

## Steps
1. Pre-organize your images into `bird/` and `drone/` folders on your Mac
2. Upload `Dataset_organized.zip` (~114 MB) to this notebook
3. Fine-tune MobileNetV2 with two-phase transfer learning
4. Evaluate with accuracy curves, confusion matrix, and classification report
5. Export to TensorFlow.js LayersModel format
6. Download the model artifacts

## Before you start

**1. GPU is optional.** CPU runtime works fine — training takes ~2–4 h instead of ~15 min.
   If you want GPU: `Runtime → Change runtime type → T4 GPU`

**2. Pre-organize your images on Mac** (open Terminal and run):
```bash
cd ~/Downloads
mkdir -p Dataset_organized/bird Dataset_organized/drone

# Bird images from valid/ and test/ splits
cp "Dataset/valid/images/B"*.jpg Dataset_organized/bird/
cp "Dataset/test/images/B"*.jpg  Dataset_organized/bird/

# Drone images from valid/ and test/ splits
cp "Dataset/valid/images/D"*.jpg Dataset_organized/drone/
cp "Dataset/test/images/D"*.jpg  Dataset_organized/drone/

# Zip the organized folder (~114 MB — uploads quickly)
zip -r Dataset_organized.zip Dataset_organized/
```

This creates `~/Downloads/Dataset_organized.zip` with two subfolders:
```
Dataset_organized/
├── bird/    ← all bird images (BV*.jpg + BT*.jpg)
└── drone/   ← all drone images (DV*.jpg + DT*.jpg)
```

> **Why valid + test only?** Together they are ~114 MB (vs 1.2 GB for the full dataset).
> MobileNetV2 is pretrained on ImageNet — ~2,600 images is sufficient for binary transfer learning.

**3. Upload `Dataset_organized.zip`** when Cell 3 shows the file picker.

In [ ]:
# Install dependencies
# tensorflowjs: converts Keras SavedModel -> TF.js format
# matplotlib, seaborn, scikit-learn: training evaluation plots
!pip install -q tensorflowjs matplotlib seaborn scikit-learn

In [ ]:
# Upload Dataset_organized.zip from your Mac
# When the file picker appears, select ~/Downloads/Dataset_organized.zip
from google.colab import files
import os, zipfile

print('A file picker will appear. Select your Dataset_organized.zip file.')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'\nUploaded: {zip_name} ({os.path.getsize(zip_name):,} bytes)')

EXTRACT_DIR = '/content/raw_dataset'
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(EXTRACT_DIR)

# The zip contains Dataset_organized/bird/ and Dataset_organized/drone/
# Find the folder that has bird/ and drone/ subfolders
TRAIN_DIR = None
for root, dirs, _ in os.walk(EXTRACT_DIR):
    if 'bird' in dirs and 'drone' in dirs:
        TRAIN_DIR = root
        break

if TRAIN_DIR is None:
    raise RuntimeError(
        'Could not find bird/ and drone/ subfolders in the uploaded zip.\n'
        'Make sure you zipped the Dataset_organized/ folder which contains bird/ and drone/ directly.'
    )

bird_count  = len([f for f in os.listdir(f'{TRAIN_DIR}/bird')  if f.lower().endswith(('.jpg','.jpeg','.png'))])
drone_count = len([f for f in os.listdir(f'{TRAIN_DIR}/drone') if f.lower().endswith(('.jpg','.jpeg','.png'))])

print(f'\nTraining directory : {TRAIN_DIR}')
print(f'Bird images        : {bird_count:,}')
print(f'Drone images       : {drone_count:,}')
print(f'Total              : {bird_count + drone_count:,}')
print('\nReady for training.')

In [ ]:
# Verify dataset structure
# TRAIN_DIR was set in Cell 3 — it already contains bird/ and drone/ subfolders.
# This cell is informational: confirm the folder structure looks correct before training.

import glob

bird_images  = glob.glob(f'{TRAIN_DIR}/bird/**/*', recursive=True)
drone_images = glob.glob(f'{TRAIN_DIR}/drone/**/*', recursive=True)

bird_images  = [f for f in bird_images  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
drone_images = [f for f in drone_images if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print('Dataset structure:')
print(f'  {TRAIN_DIR}/')
print(f'    bird/   → {len(bird_images):,} images')
print(f'    drone/  → {len(drone_images):,} images')
print(f'  Total     → {len(bird_images) + len(drone_images):,} images')

assert len(bird_images) > 0 and len(drone_images) > 0, \
    'No images found! Check that bird/ and drone/ folders are not empty.'

print('\nStructure verified. Proceeding to training.')

In [ ]:
# Configuration constants
# TRAIN_DIR is already set from Cell 3.

IMG_SIZE = 224          # MobileNetV2 expected input resolution
BATCH_SIZE = 32
EPOCHS_FROZEN = 5       # Phase 1: train head only (backbone frozen)
EPOCHS_FINETUNE = 10    # Phase 2: fine-tune last 30 backbone layers
LEARNING_RATE = 1e-4
FINETUNE_LR = 1e-5

# H5 format: compatible with tensorflowjs_converter --input_format=keras
# (TF SavedModel format is not supported for tfjs_layers_model output)
MODEL_SAVE_PATH = '/content/model.h5'
TFJS_OUTPUT_PATH = '/content/tfjs_model'

print(f'Training directory : {TRAIN_DIR}')
print(f'Input size         : {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size         : {BATCH_SIZE}')
print(f'Model save path    : {MODEL_SAVE_PATH}')

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# MobileNetV2 preprocessing maps [0, 255] -> [-1, 1]
# IMPORTANT: The React app's lib/detector.ts uses .div(127.5).sub(1)
# which is exactly the same transformation. They must stay in sync.
preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_fn,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.15,
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_fn,
    validation_split=0.2,
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42,
)

val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42,
)

CLASS_INDICES = train_generator.class_indices
CLASS_NAMES = list(CLASS_INDICES.keys())

print(f'Class indices : {CLASS_INDICES}')
print(f'Class names   : {CLASS_NAMES}')
print()
# CRITICAL: must be {'bird': 0, 'drone': 1} (alphabetical)
# If different, update CLASSES in lib/detector.ts to match!
assert 'bird' in CLASS_INDICES and 'drone' in CLASS_INDICES, \
    'Expected classes: bird and drone. Check ORGANIZED_DIR structure.'
print(f'Training samples   : {train_generator.samples:,}')
print(f'Validation samples : {val_generator.samples:,}')

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

# Load MobileNetV2 pretrained on ImageNet, without the classification head
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # Freeze backbone for Phase 1

# Build classification head
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print(f'Total parameters     : {model.count_params():,}')
print(f'Trainable parameters : {sum(v.numpy().size for v in model.trainable_variables):,}')

In [ ]:
# Phase 1: Train the classification head with the backbone frozen.
# Establishes good initial weights before fine-tuning.
print(f'Phase 1: Training head only for up to {EPOCHS_FROZEN} epochs...')

history_frozen = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FROZEN,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=3,
            restore_best_weights=True,
        ),
    ],
)

print(f'Phase 1 complete. Best val_accuracy: {max(history_frozen.history["val_accuracy"]):.4f}')

In [ ]:
# Phase 2: Unfreeze the last 30 layers and fine-tune with a lower LR
# to avoid destroying the pretrained ImageNet features.
print('Phase 2: Fine-tuning last 30 backbone layers...')

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(FINETUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINETUNE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=4,
            restore_best_weights=True,
        ),
    ],
)

print(f'Phase 2 complete. Best val_accuracy: {max(history_finetune.history["val_accuracy"]):.4f}')

In [ ]:
import matplotlib.pyplot as plt

# Combine training histories from both phases
acc = history_frozen.history['accuracy'] + history_finetune.history['accuracy']
val_acc = history_frozen.history['val_accuracy'] + history_finetune.history['val_accuracy']
loss = history_frozen.history['loss'] + history_finetune.history['loss']
val_loss = history_frozen.history['val_loss'] + history_finetune.history['val_loss']
finetune_start = len(history_frozen.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MobileNetV2 Training — Bird vs Drone Classifier', fontsize=14)

axes[0].plot(acc, label='Train Accuracy', color='royalblue')
axes[0].plot(val_acc, label='Val Accuracy', color='darkorange')
axes[0].axvline(finetune_start - 1, color='gray', linestyle='--', label='Fine-tune start')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss, label='Train Loss', color='royalblue')
axes[1].plot(val_loss, label='Val Loss', color='darkorange')
axes[1].axvline(finetune_start - 1, color='gray', linestyle='--', label='Fine-tune start')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: /content/training_curves.png')

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

val_generator.reset()
y_pred_proba = model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = val_generator.classes

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: /content/confusion_matrix.png')

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Save model in Keras H5 format.
# H5 (.h5 extension) is inferred automatically — no save_format argument needed.
# This is the format compatible with: tensorflowjs_converter --input_format=keras
#   → tfjs_layers_model, loadable via tf.loadLayersModel() in the browser.
#
# NOTE: model.export() / save_format='tf' (SavedModel) was removed because
#   tensorflowjs_converter does NOT support: tf_saved_model → tfjs_layers_model
model.save(MODEL_SAVE_PATH)
print(f'Model saved to: {MODEL_SAVE_PATH}')
!ls -lh {MODEL_SAVE_PATH}

In [ ]:
import subprocess

# Convert Keras H5 model → TF.js LayersModel format.
# --input_format=keras     : reads model.h5 (Keras H5 / SavedModel .keras)
# --output_format=tfjs_layers_model : loaded by tf.loadLayersModel() in useTFModel.ts
#
# Supported mapping (tensorflowjs_converter):
#   keras (H5)        → tfjs_layers_model  ✓  (this cell)
#   tf_saved_model    → tfjs_graph_model   ✓  (requires loadGraphModel — not used here)
#   tf_saved_model    → tfjs_layers_model  ✗  (unsupported combination — previous error)
result = subprocess.run(
    [
        'tensorflowjs_converter',
        '--input_format=keras',
        '--output_format=tfjs_layers_model',
        MODEL_SAVE_PATH,      # /content/model.h5
        TFJS_OUTPUT_PATH,     # /content/tfjs_model
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('tensorflowjs_converter failed')

print('TF.js model files:')
for f in sorted(os.listdir(TFJS_OUTPUT_PATH)):
    size = os.path.getsize(f'{TFJS_OUTPUT_PATH}/{f}')
    print(f'  {f:40s} {size:>10,} bytes')

print(f'\nExported to: {TFJS_OUTPUT_PATH}')

In [ ]:
import zipfile
from google.colab import files as colab_files

# Zip the TF.js model directory
tfjs_zip = '/content/tfjs_model.zip'
with zipfile.ZipFile(tfjs_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(TFJS_OUTPUT_PATH):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, TFJS_OUTPUT_PATH)
            zf.write(filepath, arcname)

print(f'tfjs_model.zip : {os.path.getsize(tfjs_zip):,} bytes')

# Download to your Mac
colab_files.download(tfjs_zip)
colab_files.download('/content/training_curves.png')
colab_files.download('/content/confusion_matrix.png')

print()
print('=' * 60)
print('NEXT STEPS')
print('=' * 60)
print('1. Unzip tfjs_model.zip')
print('   -> Copy model.json + *.bin into:')
print('      Airport-Bird-Drone-Detector/public/model/')
print()
print('2. The video is already in public/video/ — no action needed.')
print()
print('3. npm run dev  ->  open http://localhost:3000  ->  click Play')